In [4]:
import pandas as pd

file_path = 'data_raw/city_income.csv' 

# === 核心修改：加了 skiprows=3 ===
# 意思是：闭着眼睛跳过前3行废话，从第4行（也就是真正的表头）开始读！
# (注意：如果是跳过前2行，就改成 skiprows=2，你可以根据实际情况微调)
df_income = pd.read_csv(file_path, encoding='gbk', skiprows=3)

# === 打印预览 ===
print("=== 数据前 5 行预览 ===")
print(df_income.head())

print("\n=== 数据的列名 ===")
print(df_income.columns)

=== 数据前 5 行预览 ===
     地区  2025年    2024年    2023年    2022年    2021年    2020年    2019年    2018年  \
0    北京    NaN  6372.68  6181.10  5714.36  5932.31  5483.89  5817.10  5785.92   
1    天津    NaN  2134.20  2027.51  1846.69  2141.06  1923.11  2410.41  2106.24   
2   石家庄    NaN   746.08   737.89   692.16   681.36   632.19   569.13   519.68   
3    太原    NaN   440.26   449.24   437.48   423.44   378.44   386.62   373.23   
4  呼和浩特    NaN   255.12   238.06   230.87   228.92   217.10   203.12   204.72   

     2017年  ...    2015年    2014年    2013年    2012年    2011年    2010年  \
0  5430.79  ...  4723.86  4027.16  3661.11  3314.93  3006.28  2353.93   
1  2310.36  ...  2667.11  2390.35  2079.07  1760.02  1455.13  1068.81   
2   460.89  ...   375.05   343.47   315.12   272.28   221.23   163.63   
3   311.85  ...   274.24   258.85   247.33   215.67   174.72   138.48   
4   201.50  ...   247.40   211.54   182.02   178.64   151.43   126.76   

     2009年    2008年    2007年    2006年  
0  2026.81  1837

In [5]:
# === 第一步：将“宽表”融化(melt)成“长表” ===
# id_vars: 保持不动的列（城市）
# var_name: 那些被折叠的表头（年份）变成一列，取名叫 'year'
# value_name: 里面的具体数据变成一列，取名叫 'income' (对应老师要求的变量名)
df_income_long = pd.melt(df_income, 
                         id_vars=['地区'], 
                         var_name='year', 
                         value_name='income')

# === 第二步：改名与清洗（学术规范） ===
# 1. 按照老师的要求，把'地区'改名为'city'
df_income_long.rename(columns={'地区': 'city'}, inplace=True)

# 2. 清洗年份：把'2024年'里的'年'字替换掉，并变成整数型(int)，方便以后画图和合并
df_income_long['year'] = df_income_long['year'].str.replace('年', '').astype(int)

# 3. 把没有数据的行（比如2025年 NaN）删掉
df_income_long = df_income_long.dropna(subset=['income'])

# === 第三步：看看魔法效果 ===
# 按照城市和年份排个序，强迫症福音
df_income_long = df_income_long.sort_values(by=['city', 'year'], ascending=[True, False]).reset_index(drop=True)

print("=== 见证奇迹：清洗后的长表面板数据 ===")
print(df_income_long.head(10))

=== 见证奇迹：清洗后的长表面板数据 ===
  city  year   income
0   上海  2024  8374.17
1   上海  2023  8312.50
2   上海  2022  7608.00
3   上海  2021  7771.80
4   上海  2020  7046.30
5   上海  2019  7165.10
6   上海  2018  7108.15
7   上海  2017  6642.26
8   上海  2016  6406.13
9   上海  2015  5519.50


In [6]:
import pandas as pd

# === 第一步：建造“自动化清洗流水线” (定义函数) ===
# 这个函数接收两个参数：文件名(file_name) 和 你想给指标起的英文名(col_name)
def clean_stats_data(file_name, col_name):
    # 1. 自动拼接相对路径
    path = f'data_raw/{file_name}'
    
    # 2. 读取数据 (跳过前3行，用gbk解码)
    df = pd.read_csv(path, encoding='gbk', skiprows=3)
    
    # 3. 宽表转长表
    df_long = pd.melt(df, id_vars=['地区'], var_name='year', value_name=col_name)
    
    # 4. 规范化命名和数据类型
    df_long.rename(columns={'地区': 'city'}, inplace=True)
    df_long['year'] = df_long['year'].str.replace('年', '').astype(int)
    
    # 5. 去除空值并返回清洗好的干干净净的数据
    df_long = df_long.dropna(subset=[col_name])
    return df_long

# === 第二步：把文件扔进流水线 ===
print("开始清洗各个子表...")
df_income = clean_stats_data('city_income.csv', 'income')
df_expend = clean_stats_data('city_expenditure.csv', 'expend')
df_gdp = clean_stats_data('gdp.csv', 'gdp')

# 如果你还需要住户存款数据，也可以随时加上：
# df_deposit = clean_stats_data('individual_deposit.csv', 'deposit')

# === 第三步：多表合并 (超级 VLOOKUP) ===
print("正在将表格进行合并...")
# 先把 收入 和 支出 合并
df_master = pd.merge(df_income, df_expend, on=['city', 'year'], how='inner')
# 再把 GDP 拼上去
df_master = pd.merge(df_master, df_gdp, on=['city', 'year'], how='inner')

# === 第四步：完成老师布置的计算任务 ===
# 在 Python 里做列计算，比 Excel 还简单，直接两列相减/相除即可：
# 1. 计算预算缺口 gap = expend - income
df_master['gap'] = df_master['expend'] - df_master['income']

# 2. 计算缺口占GDP比重 gap_to_gdp = gap / gdp
df_master['gap_to_gdp'] = df_master['gap'] / df_master['gdp']

# 按城市和年份排个序，强迫症最爱
df_master = df_master.sort_values(by=['city', 'year'], ascending=[True, False]).reset_index(drop=True)

print("\n=== 大功告成！最终的宏观面板数据集 ===")
print(df_master.head(10))

# === 第五步：保存清洗成果 ===
# 存入我们之前建好的 data_clean 文件夹中，供后续分析使用
# index=False 的意思是保存的时候不要把最左边的 0,1,2,3 行号写进文件里
df_master.to_csv('data_clean/city_master_data.csv', index=False, encoding='utf-8-sig')
print("\n✅ 数据已成功保存到 data_clean/city_master_data.csv")

开始清洗各个子表...
正在将表格进行合并...

=== 大功告成！最终的宏观面板数据集 ===
  city  year   income   expend      gdp      gap  gap_to_gdp
0   上海  2024  8374.17  9874.84  53759.5  1500.67    0.027915
1   上海  2023  8312.50  9638.51  51404.5  1326.01    0.025796
2   上海  2022  7608.00  9393.00  48594.5  1785.00    0.036733
3   上海  2021  7771.80  8430.86  47059.4   659.06    0.014005
4   上海  2020  7046.30  8102.11  41603.9  1055.81    0.025378
5   上海  2019  7165.10  8179.28  40241.2  1014.18    0.025203
6   上海  2018  7108.15  8351.54  37769.1  1243.39    0.032921
7   上海  2017  6642.26  7547.62  34378.3   905.36    0.026335
8   上海  2016  6406.13  6918.94  30963.9   512.81    0.016562
9   上海  2015  5519.50  6191.56  27821.6   672.06    0.024156

✅ 数据已成功保存到 data_clean/city_master_data.csv


In [7]:
# === 避坑第一步：按时间正序排列 ===
# 必须保证年份是 2006, 2007, 2008... 这样顺着排，增长率才能算对！
df_master = df_master.sort_values(by=['city', 'year'], ascending=[True, True])

# === 第二步：分组计算增长率 (核心魔法) ===
# groupby('city'): 按城市分组
# ['income']: 提取收入这一列
# pct_change(): 自动计算 (本期-上期)/上期
df_master['income_growth'] = df_master.groupby('city')['income'].pct_change()
df_master['expend_growth'] = df_master.groupby('city')['expend'].pct_change()

# === 第三步：恢复你喜欢的倒序排列（强迫症福音）===
df_master = df_master.sort_values(by=['city', 'year'], ascending=[True, False]).reset_index(drop=True)

# 打印看看结果 (注意看2024年的数据后面，是不是多了两个growth列)
print("=== 带有增长率的最终面板数据 ===")
print(df_master[['city', 'year', 'income', 'income_growth', 'expend', 'expend_growth']].head(10))

# 既然数据又丰富了，我们把刚才保存的干净数据覆盖更新一下
df_master.to_csv('data_clean/city_master_data.csv', index=False, encoding='utf-8-sig')
print("\n✅ 数据已更新并保存！")

=== 带有增长率的最终面板数据 ===
  city  year   income  income_growth   expend  expend_growth
0   上海  2024  8374.17       0.007419  9874.84       0.024519
1   上海  2023  8312.50       0.092600  9638.51       0.026138
2   上海  2022  7608.00      -0.021076  9393.00       0.114121
3   上海  2021  7771.80       0.102962  8430.86       0.040576
4   上海  2020  7046.30      -0.016580  8102.11      -0.009435
5   上海  2019  7165.10       0.008012  8179.28      -0.020626
6   上海  2018  7108.15       0.070140  8351.54       0.106513
7   上海  2017  6642.26       0.036860  7547.62       0.090864
8   上海  2016  6406.13       0.160636  6918.94       0.117479
9   上海  2015  5519.50       0.203672  6191.56       0.257568

✅ 数据已更新并保存！
